In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from scipy.io import loadmat

from tqdm import tqdm
import nibabel as nib
import nilearn.image as nl_image

from nsdcode.nsd_mapdata import NSDmapdata #https://github.com/cvnlab/nsdcode.git

import h5py
from PIL import Image



In [ ]:
SUBJECT_IDS = list(range(1, 9))
# SUBJECT_IDS = [SUBJECT_IDS[0]]  # For testing purposes, only use the first subject

SUBJECTS = [
    f"subj{i:02d}" for i in SUBJECT_IDS
]

In [ ]:
ROI_FILES = [
    "streams",
    "prf-visualrois",
    "nsdgeneral",
    "floc-words",
    "floc-places",
    "floc-faces",
    "floc-bodies"
]

In [ ]:
ds_dir = '${MBS_NSD_DIR}'

ds_dir = Path(ds_dir)

In [ ]:
nsd = NSDmapdata(base_dir=ds_dir)

In [ ]:
for sub_id in tqdm(SUBJECT_IDS):
    output_dir = ds_dir / "nsddata" / "freesurfer" / "fsaverage" / "mapped_labels" / f"subj{sub_id:02d}"
    if not output_dir.exists():
        output_dir.mkdir(parents=True)

    nativesurface_roi_dir = ds_dir / "nsddata" / "freesurfer" / f"subj{sub_id:02d}" / "label"
    
    # Copy {nsdgeneral, streams} ROIs from fsaverage/label
    for roi_file in ['nsdgeneral', 'streams']:
        for hemi in ['lh', 'rh']:
            fsaverage_roi_path = ds_dir / "nsddata" / "freesurfer" / "fsaverage" / "label" / f"{hemi}.{roi_file}.mgz"
            target_roi_path = output_dir / f"{hemi}.{roi_file}.mgz"
            shutil.copyfile(fsaverage_roi_path, target_roi_path)
            
    
    for roi_file in tqdm(ROI_FILES, desc=f"Mapping ROIs for {subject}", leave=False):
        for hemi in ['lh', 'rh']:
            
            native_roi_path = nativesurface_roi_dir / f"{hemi}.{roi_file}.mgz"
            target_roi_path = output_dir / f"{hemi}.{roi_file}.mgz"
            
            # Map from native surface to fsaverage
            mapped_to_fsaverage = nsd.fit(
                subjix=sub_id, 
                sourcespace=f'{hemi}.white', 
                targetspace='fsaverage', 
                sourcedata=str(native_roi_path),
                outputfile=str(target_roi_path),
                fsdir=ds_dir / "nsddata" / "freesurfer" / "fsaverage"
                )
            

        

In [ ]:
for sub_id in tqdm(SUBJECT_IDS):
    output_dir = ds_dir / "nsddata" / "mapped_labels" / f"subj{sub_id:02d}"
    
    print(f"Subject subj{sub_id:02d} mapped ROIs:")
    print("\t", [file.name for file in output_dir.glob("*.mgz")], "\n")